# 07 — Operating it in production

**What you will learn**

- The complete error taxonomy — 400, 413, 501, 503, 504 — and what each one
  tells you to *do*, which is different for every one of them.
- Why size limits split between `400` and `413`.
- A retry client that backs off correctly, and why fanning out concurrency
  instead is the wrong instinct.
- What to monitor, and why `/health` alone is not a monitor.
- Managing a model swap without corrupting your dataset.
- A pre-production checklist drawn from everything in this path.

**What it assumes you already did**

All six earlier notebooks. This one ties them together, and it references the
503 mechanics from [06](06-batching-and-throughput.ipynb) and the response-shape
contract from [04](04-tuning-and-response-shapes.ipynb) directly.

**Roughly how long**

About 25 minutes.

## Setup

In [1]:
import json
import os
import time

import requests

# Every notebook in this path reads the same environment variable, so you can
# point the whole series at a different deployment with one export:
#     export GLINER_BASE_URL=http://localhost:8013
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# The server bounds its own inference at REQUEST_TIMEOUT_SECONDS (default 120)
# and returns 504 when it blows through that. A client timeout slightly above
# the server's means the server always gets to explain itself with a status
# code instead of the client giving up first and leaving you guessing.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON. Raises on non-2xx."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises.

    Used whenever the interesting part of the answer IS the status code.
    """
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    """Pretty-print a JSON-serializable object."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

BASE_URL = http://192.168.1.177:8013


## The error taxonomy

The service bounds everything and returns defined status codes with a
`{"detail": ...}` body. These are not edge cases to swallow in a bare `except` —
each one names a **different corrective action**, and treating them uniformly
throws away the most useful information the service gives you.

| Status | Trigger | What *you* should do |
|---|---|---|
| **400** | Unknown key in the payload or in `schema_config` | Fix the client. The detail lists the allowed keys |
| **400** | Malformed payload: missing/empty `text`/`texts`, bad `labels` type, missing `schema_config` | Fix the client |
| **400** | Bad inference option: `threshold` outside `[0,1]`, non-boolean `include_spans`, non-positive `max_len`/`batch_size` | Fix the client |
| **400** | Too many labels or schema fields (`MAX_LABELS`, `MAX_SCHEMA_FIELDS`) | Reduce the schema |
| **413** | `text` over `MAX_TEXT_CHARS` | Split the document client-side |
| **413** | `texts` over `MAX_BATCH_SIZE`, or batch chars over `MAX_BATCH_CHARS` | Re-chunk the batch (notebook 06) |
| **501** | Boundary-only route on a span checkpoint | Change `MODEL_ID` and restart; this is a deployment fact, not a request problem |
| **503** | No inference slot within `INFERENCE_ACQUIRE_TIMEOUT_SECONDS` | **Retry with backoff.** Do not add concurrency |
| **503** | `/health/deep` probe failed — body carries `"status": "degraded"` | Page someone; the model is not answering |
| **504** | Inference exceeded `REQUEST_TIMEOUT_SECONDS` | Split the document. The request was too big for the budget, the service is not down |
| **500** | Inference raised inside the model | A bug or a resource problem; check logs |

The single most important distinction in that table: **400/413/501 are your
fault and retrying is pointless; 503/504 are transient or sizing problems and
retrying may work.** A client that retries a 400 five times has turned one clear
error into five, and learned nothing.

### Why the 400/413 split on size

Look at where the line falls. Label counts and schema field counts are `400`.
Characters and document counts are `413`.

That is not arbitrary. `413 Payload Too Large` is specifically about the size of
the request *entity* — the bytes you sent. A 20 001-character document really is
an oversized payload, and the fix is genuinely client-side chunking of content.

Sending 300 labels is not an oversized payload; it is a **badly formed request**
under this service's contract. The bytes are trivial. What is wrong is the
schema you asked for, and the fix is to ask for a smaller schema, not to split
your input.

The practical value of the split is that it lets a client route by status code
alone: `413` means "my chunking logic is wrong, re-chunk and retry", `400` means
"my schema or payload is wrong, do not retry". Those are different code paths,
and the split lets you dispatch to them without parsing the message.

In [2]:
checks = [
    ("400 missing 'text'",
     "/extract_entities", {"labels": ["person"]}),
    ("400 'labels' as a string",
     "/extract_entities", {"text": "Tim Cook.", "labels": "person"}),
    ("400 typo'd inference option",
     "/extract_entities", {"text": "Tim Cook.", "labels": ["person"], "treshold": 0.5}),
    ("400 typo'd schema_config key",
     "/extract_multitask", {"text": "Tim Cook.", "schema_config": {"entitys": ["person"]}}),
    ("400 task key at top level",
     "/extract_multitask", {"text": "Tim Cook.", "entities": ["person"]}),
    ("400 threshold out of range",
     "/extract_entities", {"text": "Tim Cook.", "labels": ["person"], "threshold": 1.5}),
    ("400 classify with a bare list",
     "/classify_text", {"text": "Great product.", "labels": ["positive", "negative"]}),
    ("413 oversized 'text'",
     "/extract_entities", {"text": "a" * 20_001, "labels": ["x"]}),
    ("413 batch over MAX_BATCH_SIZE",
     "/extract_entities_batch", {"texts": ["hello"] * 65, "labels": ["person"]}),
    ("413 batch over MAX_BATCH_CHARS",
     "/extract_entities_batch", {"texts": ["a" * 19_000] * 15, "labels": ["person"]}),
]

for name, path, payload in checks:
    status, body = post_raw(path, payload)
    detail = body.get("detail") if isinstance(body, dict) else body
    print(f"{name:<34} HTTP {status}")
    print(f"    {detail}")

400 missing 'text'                 HTTP 400
    Provide non-empty 'text'.
400 'labels' as a string           HTTP 400
    Provide 'labels' as list or dict.
400 typo'd inference option        HTTP 400
    Unknown payload key(s): ['treshold']. Allowed: ['include_confidence', 'include_spans', 'labels', 'max_len', 'overlap_policy', 'text', 'threshold'].
400 typo'd schema_config key       HTTP 400
    Unknown schema_config key(s): ['entitys']. Allowed: ['classification', 'entities', 'relations', 'structure'].
400 task key at top level          HTTP 400
    Unknown payload key(s): ['entities']. Allowed: ['include_confidence', 'include_spans', 'max_len', 'overlap_policy', 'schema_config', 'text', 'threshold'].
400 threshold out of range         HTTP 400
    'threshold' must be in [0, 1].
400 classify with a bare list      HTTP 400
    'labels' must be an object mapping a task name to its labels, e.g. {"sentiment": ["positive", "negative"]}. A bare list is not accepted.


413 oversized 'text'               HTTP 413
    'text' exceeds MAX_TEXT_CHARS=20000.


413 batch over MAX_BATCH_SIZE      HTTP 413
    'texts' exceeds MAX_BATCH_SIZE=64.
413 batch over MAX_BATCH_CHARS     HTTP 413
    batch totals 285000 chars, exceeding MAX_BATCH_CHARS=40000. Split the batch.


Notice how much the `Allowed:` lists give you. They are generated from the
route's own key set, so they are always correct for the build you are actually
talking to — more reliable than any documentation, this notebook included. When
a client breaks after an upgrade, read that list first.

## Retrying correctly

`503` is **backpressure**, not failure. The server is telling you its single
inference slot was not free within the acquire timeout. On a one-GPU box under
load, that is designed behavior — the alternative is unbounded queueing, where
requests pile up until something times out much later with no useful signal.

The correct response is retry with **exponential backoff and jitter**.

Backoff, because the condition is temporary and hammering it immediately just
burns the retry budget while the slot is still busy.

Jitter, because without it every client that got a 503 at the same moment will
retry at the same moment. Synchronized retries produce a thundering herd that
reproduces the exact condition that caused the 503 — you get a self-sustaining
oscillation instead of recovery. A small random offset decorrelates them.

`504` gets the same treatment structurally but for a different reason: it means
inference exceeded the budget. Retrying an identical oversized document will
just time out again, so a 504 that repeats is a signal to **split the input**,
not to keep retrying. The client below retries both, which is right for
transient 504s, but persistent 504s need a code change.

**What you should not do** is respond to 503 by increasing concurrency. That is
the natural instinct — "requests are being rejected, so send more" — and
notebook 06 measured why it fails: throughput went from 8.0 to 3.4 docs/s
raising `MAX_CONCURRENT_INFERENCES` from 1 to 8. The GPU serializes regardless.
More concurrent requests means more requests waiting on the same lock, more
of them hitting the acquire timeout, and more peak memory. The correct
throughput lever is batching.

In [3]:
import random


def post_with_retry(path, payload, attempts=5, base_delay=0.5):
    """POST with exponential backoff + jitter on 503 (busy) and 504 (timeout).

    Deliberately does NOT retry 400/413/501 - those are client or deployment
    problems and a retry is guaranteed to fail the same way.
    """
    for attempt in range(attempts):
        r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)

        if r.status_code not in (503, 504):
            r.raise_for_status()      # 400/413/501 raise immediately - no retry
            return r.json()

        if attempt == attempts - 1:
            r.raise_for_status()

        delay = base_delay * (2 ** attempt) + random.uniform(0, base_delay)
        print(f"  HTTP {r.status_code}, retrying in {delay:.2f}s "
              f"(attempt {attempt + 1}/{attempts})")
        time.sleep(delay)


show(post_with_retry("/extract_entities", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "labels": ["company", "person", "location"],
}))

{
  "entities": {
    "company": [
      "Apple"
    ],
    "person": [
      "Tim Cook"
    ],
    "location": [
      "Cupertino"
    ]
  }
}


The `r.raise_for_status()` on the non-retryable branch is the load-bearing line.
It is what stops a `400` from being retried five times and reported as a timeout
twenty seconds later. Failing fast on client errors is not just efficiency — it
preserves the diagnostic.

## Monitoring

Notebook 01 introduced the three GET routes. Here is what to actually do with
them.

**Poll `/health/deep`, not `/health`.**

`/health` reports process state from memory. A worker whose CUDA context has
wedged keeps returning `"status": "ok"` there indefinitely, because from the web
process's point of view nothing is wrong — it holds a model object, it just
cannot use it. Monitoring `/health` in that state gives you a green dashboard
over a dead service, which is worse than no monitoring at all.

`/health/deep` runs a real forward pass on every call and returns `503` with
`"status": "degraded"` when the pass fails or times out. Because failure is a
status code rather than a field in a 200 body, `curl -f` is a complete check
with no JSON parsing:

```bash
curl -fs http://192.168.1.177:8013/health/deep >/dev/null || alert
```

**And it does not lie about a busy box.** The probe deliberately bypasses the
inference semaphore. If it queued like a normal request, a sustained batch job
would make the probe time out and report `degraded` — paging you for a service
that was merely working hard. The bypass is what keeps "wedged" and "busy"
distinguishable, and those need opposite responses: one needs a restart, the
other needs you to leave it alone.

Measured on `jarvita-agx`: while a 48-document batch held the only inference
slot, `/health/deep` returned `200` in 1.56 s while a normal request queued
4.08 s. Single measurement — it demonstrates the bypass works, not a latency
SLO.

**What to alert on and what to trend:**

| Signal | Source | Use |
|---|---|---|
| `/health/deep` non-200 | status code | **Alert.** The model is not answering |
| `probe.latency_ms` | `/health/deep` body | Trend. Model-health latency, uncontaminated by queueing |
| `saturated` / `inflight` | `/health` | Trend. Load level — busy is not an alert |
| `503` rate at the client | your client | Trend. Rising rate means batch harder, not scale up |
| `loaded` | `/health` | Alert if `false` long after startup |
| `device != "cuda"` | `/health` | **Alert.** On a Jetson this is a fault, not a mode |

Note what is deliberately *not* an alert: `saturated: true` and a nonzero `503`
rate. Both are the system working as designed under load. Alerting on them
trains people to ignore the dashboard.

In [4]:
r = session.get(f"{BASE_URL}/health/deep", timeout=TIMEOUT)
deep = r.json()

print("HTTP", r.status_code, "| status:", deep["status"])
print("probe latency :", deep.get("probe", {}).get("latency_ms"), "ms")
print("device        :", deep.get("device"))
print("inflight      :", deep.get("inflight"), "| saturated:", deep.get("saturated"))
print("loaded        :", deep.get("loaded"))

# `curl -f` semantics in Python: the status code alone is the whole check.
print("\nservice healthy:", r.status_code == 200)

HTTP 200 | status: ok
probe latency : 350.2 ms
device        : cuda
inflight      : 1 | saturated: True
loaded        : True

service healthy: True


## Model swaps without corrupting your dataset

Changing `MODEL_ID` is a one-line config change and a restart. Its consequences
are considerably larger than that suggests, and this is the failure mode most
likely to bite you months after deployment.

Everything in this path that depends on the checkpoint:

- **Span boundaries move.** Notebook 02's example: `"PDG de Renault"` versus
  `"Renault"` for the same sentence across two checkpoints. Any downstream exact
  string matching silently produces fewer joins. Nothing errors.
- **Architecture may change.** A span checkpoint makes `/extract_relations` and
  all four batch routes return `501`. Your bulk pipeline stops working entirely
  — which is at least a loud failure.
- **Confidence distributions shift**, so a threshold tuned on one checkpoint is
  not tuned on the next (notebook 04).
- **Latency and memory change.** Single-document latency measured on this box,
  average of 5 runs over six sentences: `gliner2.5-base-v1` 83 ms / 748 MB,
  `gliner2.5-multi-v1` 94 ms / 1.1 GB, `gliner2-large-v1` 109 ms / 1.9 GB.
  Shared box, small sample — indicative only.

The defensive measure is the one from notebook 01: **stamp
`/version.model_revision` onto every row you persist.** It costs one column and
it is the only thing that makes two extraction populations separable after the
fact. Without it, a model swap quietly merges two models' opinions into one
table and no analysis downstream can tell them apart.

The cell below builds a provenance stamp you can attach to output.

In [5]:
version = get("/version")
health = get("/health")

PROVENANCE = {
    "model_id": version["model_id"],
    "model_revision": version["model_revision"],
    "architecture": version["architecture"],
    "gliner2": version["gliner2"],
    "torch": version["torch"],
}
show(PROVENANCE)

result = post("/extract_entities", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "labels": ["company", "person", "location"],
})

row = {"entities": result["entities"], **PROVENANCE, "extracted_at": time.time()}
print("\na persistable row:")
show(row)

{
  "model_id": "fastino/gliner2.5-base-v1",
  "model_revision": "4a3138e2432c24b4",
  "architecture": "boundary",
  "gliner2": "2.0.0",
  "torch": "2.8.0"
}



a persistable row:
{
  "entities": {
    "company": [
      "Apple"
    ],
    "person": [
      "Tim Cook"
    ],
    "location": [
      "Cupertino"
    ]
  },
  "model_id": "fastino/gliner2.5-base-v1",
  "model_revision": "4a3138e2432c24b4",
  "architecture": "boundary",
  "gliner2": "2.0.0",
  "torch": "2.8.0",
  "extracted_at": 1788616527.793716
}


Fetch `/version` **once at startup**, not per request — it is process metadata,
not per-call state, and a model cannot change under a running container. One
call, cached, stamped onto everything.

There is one caveat worth naming: if you run several replicas behind a load
balancer, they could in principle be mid-rollout on different checkpoints. In
that case the per-process cached value is still correct for the rows *that
process* produced, which is exactly what you want, and is a good argument for
stamping rather than recording the model once in a job's metadata.

## Try this yourself

Write a bulk-ingest function that puts the whole path together, then deliberately
break it and watch which errors are retryable.

The cell below runs a small corpus through the batch route with retries and
provenance. Before running it, predict what happens on the third call — the one
with a deliberately oversized document.

In [6]:
def ingest(texts, schema_config, max_chars=40_000, max_size=64):
    """Batch-ingest with chunking, retry, provenance, and honest error routing."""
    rows, failures = [], []
    batch, chars = [], 0

    def flush(b):
        if not b:
            return
        try:
            r = post_with_retry("/extract_multitask_batch",
                                {"texts": b, "schema_config": schema_config})
            for text, res in zip(b, r["results"]):
                rows.append({"text": text, "result": res, **PROVENANCE})
        except requests.HTTPError as e:
            code = e.response.status_code
            retryable = code in (503, 504)
            failures.append({"n": len(b), "status": code,
                             "retryable": retryable,
                             "detail": e.response.json().get("detail")})

    for t in texts:
        if batch and (len(batch) >= max_size or chars + len(t) > max_chars):
            flush(batch)
            batch, chars = [], 0
        batch.append(t)
        chars += len(t)
    flush(batch)
    return rows, failures


CORPUS = [
    "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "Satya Nadella leads Microsoft from Redmond.",
    "a" * 25_000,   # deliberately over MAX_TEXT_CHARS
]

SCHEMA = {
    "entities": ["company", "person", "location"],
    "classification": {"name": "sentiment", "labels": ["positive", "negative"]},
}

rows, failures = ingest(CORPUS, SCHEMA)
print(f"{len(rows)} row(s) ingested, {len(failures)} batch failure(s)\n")
for f in failures:
    print("FAILED:", f)
if rows:
    print("\nfirst row:")
    show(rows[0])

0 row(s) ingested, 1 batch failure(s)

FAILED: {'n': 3, 'status': 413, 'retryable': False, 'detail': "'text' exceeds MAX_TEXT_CHARS=20000."}


**Discussion.** The oversized document does not produce a `413` on the batch
size or character bound — it produces a `413` on `MAX_TEXT_CHARS`, because a
single document over the per-document limit is rejected regardless of how small
the batch around it is. That is notebook 06's edge case: **the chunker cannot
save you from an individual document that is too long.** It is a different
problem with a different fix (split the document, or drop it), and the chunker
correctly refuses to paper over it.

Notice too that the failure is recorded as `retryable: False`. That is the whole
point of routing by status code rather than catching everything: this batch will
never succeed no matter how many times you send it, so a retry loop would burn
the budget and then report a misleading timeout. Meanwhile a genuine `503` in
the same run would be retried transparently by `post_with_retry` and never reach
the failure list at all.

Two things this toy function does that a real one should keep:

- **Failures carry the detail string**, so the operator sees `'text' exceeds
  MAX_TEXT_CHARS=20000.` in the log rather than "batch 3 failed".
- **Provenance is attached per row**, not per job, so a mid-run restart onto a
  different checkpoint stays visible in the data.

And one thing to add for real use: `max_chars` is hardcoded above for clarity,
but notebook 06 showed how to discover the live bound from a `413` detail at
startup. Do that instead — the code default and this deployment's configured
value are not the same number.

## Pre-production checklist

Everything in this path, as things to verify before you point real traffic at it.

**Client contract**

- [ ] `GLINER_BASE_URL` (or equivalent) is configuration, not a literal.
- [ ] Client timeout is above the server's `REQUEST_TIMEOUT_SECONDS`, so the
      server gets to report failures as status codes.
- [ ] `include_confidence` / `include_spans` are fixed per consumer, and the
      choice is documented as part of your stored schema. Normalization happens
      once at the boundary.
- [ ] Retries cover `503`/`504` with exponential backoff **and jitter**, and
      explicitly do not cover `400`/`413`/`501`.
- [ ] Batches are chunked against **both** `MAX_BATCH_SIZE` and
      `MAX_BATCH_CHARS`, with the live values discovered at startup rather than
      hardcoded.
- [ ] Individual documents are split against `MAX_TEXT_CHARS` before batching.

**Model contract**

- [ ] `architecture == "boundary"` is asserted at startup if you use relations
      or batch routes. Assert on `architecture`, never on `model_class`.
- [ ] `model_revision` is stamped onto every persisted row.
- [ ] Span boundaries were re-checked against your own texts on the checkpoint
      you are actually deploying.
- [ ] Relation output is type-validated against the entity set from the same
      forward pass (notebook 05).
- [ ] `threshold` is left at the default unless you verified, with
      `include_confidence` on your own data, that false positives really do
      score below true positives.

**Operations**

- [ ] Monitoring polls `/health/deep` and alerts on non-200.
- [ ] `device != "cuda"` alerts.
- [ ] `saturated` and client-side `503` rate are trended, **not** alerted.
- [ ] `probe.latency_ms` is trended.
- [ ] Nobody is planning to "fix" 503s by raising `MAX_CONCURRENT_INFERENCES`.
- [ ] Bulk paths use `/extract_multitask_batch` rather than per-document calls.
- [ ] If `MAX_BATCH_CHARS` was raised, `docker stats` was watched through a
      full-size batch first.

## What you learned

- Five status codes, five different corrective actions. 400/413/501 are not
  retryable; 503/504 are. Route by status code, not by a catch-all handler.
- Size limits split `400`/`413` by *kind*: a bad schema is a malformed request,
  an oversized document is an oversized payload. The split lets a client
  dispatch without parsing messages.
- `503` is backpressure. Retry with backoff **and jitter** — jitter prevents the
  synchronized retry storm that recreates the condition. Never respond by adding
  concurrency; measured 8.0 → 3.4 docs/s going 1 → 8 slots.
- Persistent `504` means split the document, not retry harder.
- Monitor `/health/deep` (real forward pass, semaphore-bypassing, `503` on
  failure), not `/health` (green over a wedged worker). Alert on non-200 and on
  `device != "cuda"`; trend saturation and 503 rate rather than alerting on them.
- A model swap moves span boundaries, may change architecture, and shifts
  confidence distributions — all silently. Stamp `model_revision` on every row;
  fetch `/version` once at startup.
- The chunker cannot rescue a single oversized document. That is a
  `MAX_TEXT_CHARS` problem with a different fix.

## Where to go next

You have finished the path. For reference material:

| Need | Document |
|---|---|
| All 12 endpoints, verified response bodies, full error table, configuration | [`../api.md`](../api.md) |
| Deploy, monitoring, env vars, batch sizing, model swap, rollback, failures | [`../runbook.md`](../runbook.md) |
| What GLiNER2/2.5 is, boundary-only capabilities, model comparison | [`../wiki.md`](../wiki.md) |
| The original terse reference notebook this path was expanded from | [`../examples.ipynb`](../examples.ipynb) |
| Jetson build decisions and the `sbsa/cu130` trap | [`../../JETSON.md`](../../JETSON.md) |

Live API schema, straight from the running service:

```
http://192.168.1.177:8013/docs
http://192.168.1.177:8013/redoc
http://192.168.1.177:8013/openapi.json
```

List every route the running build actually exposes:

```bash
curl -s http://192.168.1.177:8013/openapi.json \
  | python3 -c "import json,sys; d=json.load(sys.stdin); [print(m.upper(), p) for p,v in d['paths'].items() for m in v]"
```